**Homework 5: Polynomial Regression and Overfitting**

Does a more flexible model make better predictions? We will use the car displacement/MPG dataset from Homework 4 to build polynomial models and compare their training and validation errors.

You already know NumPy arrays, train/test splitting, and the normal equations. Here we build **StandardScaler**, **PolynomialFeatures**, and **LinearRegression** ourselves. No scikit-learn model or preprocessing class is used; only its familiar data-splitting utility is imported.

1. Split the observations and scale using training statistics.
2. Build power features and fit a quadratic model using the normal equations.
3. Compare degree 1, 2, and 8 curves, then training and validation MSE for degrees 1 through 8.

The code is complete. Read it, predict intermediate values, and answer the matching Machine Learning Lab questions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

We will continue with the displacement `disp` and `mpg` columns of the `cars` dataset, as in the last assignment. (This time we'll sort these by `disp` to make visualization easier later.)

In [ ]:
cars=pd.read_csv('https://vincentarelbundock.github.io/Rdatasets/csv/causaldata/auto.csv')
disp=np.array(cars.displacement)
mpg=np.array(cars.mpg)

index=np.argsort(disp)
disp=disp[index]
mpg=mpg[index]

## 1. Separate learning from evaluation

Reserve 20% of the observations before fitting anything. Because we use this held-out set to choose the polynomial degree, we call it a **validation set**, not a final test set. An independent test set would be needed for an unbiased final evaluation after choosing the degree.

The fixed random state makes the split repeatable. The split shuffles rows, so sorting the original data does not keep the returned training array sorted. Here displacement stays **one-dimensional**, as required by our PolynomialFeatures class.

In [ ]:
Xtrain, Xval, ytrain, yval = train_test_split(
    disp, mpg, test_size=0.2, random_state=42
)
len(Xtrain), len(Xval)

## 2. Scale before taking powers

Large raw displacement values become enormous when raised to high powers. The class below stores the mean and population standard deviation (NumPy's default) and uses them to scale and unscale values. For a matrix, axis=0 works column by column; here the input is a one-dimensional array.

Read how fit, transform, and inverse_transform use the same stored statistics.

In [ ]:
class StandardScaler():
    def __init__(self):
        pass

    def fit(self,X):
        self.mean=X.mean(axis=0)
        self.std=X.std(axis=0)

    def transform(self,X):
        return (X-self.mean)/self.std

    def inverse_transform(self,X):
        return X*self.std+self.mean

Fit the scaler on **Xtrain only**. Use that same scaler on Xval so the model sees both sets in the same coordinate system. Do not fit a second scaler on validation data. Scaling improves numerical behavior, but does not prevent overfitting or guarantee stable calculations at arbitrarily high degrees.

In [ ]:
disp_scaler=StandardScaler()
disp_scaler.fit(Xtrain)
scaled_Xtrain=disp_scaler.transform(Xtrain)
scaled_Xval=disp_scaler.transform(Xval)

scaled_Xtrain.mean(), scaled_Xtrain.std()

In [ ]:
disp_scaler.inverse_transform(scaled_Xtrain[:3]), Xtrain[:3]

## 3. Reuse our normal-equation model

This is the from-scratch LinearRegression class from Homework 4, not the scikit-learn class. It adds its own column of ones, stores the intercept separately, and expects a two-dimensional feature matrix. Its attributes are coef and intercept (without trailing underscores).

The explicit inverse keeps the normal-equation calculation visible. This teaching implementation assumes independent feature columns; production implementations use more numerically robust solvers.

In [ ]:
class LinearRegression():
    def __init__(self):
        pass

    def fit(self,X,y):
        '''stores the slope and intercept
        for the model defined by X and y'''
        Xnew=np.ones((X.shape[0],X.shape[1]+1))
        Xnew[:,1:]=X
        coeffs=np.linalg.inv(Xnew.T@Xnew)@(Xnew.T@y)
        self.intercept=coeffs[0]
        self.coef=coeffs[1:]

    def predict(self,x):
        '''x is expected to have shape
        (num_test_obs,num_feats)'''
        return x@self.coef+self.intercept

## 4. Build the polynomial features ourselves

For a one-dimensional array X, PolynomialFeatures builds columns containing successive powers. Without bias, degree 3 gives columns X, X**2, and X**3. With bias it also includes X**0, a column of ones.

When fitting our LinearRegression class, keep **include_bias=False**: LinearRegression already adds the constant column. Including it twice makes the normal-equation matrix singular.

In [ ]:
class PolynomialFeatures():
    def __init__(self,degree,include_bias=False):
        self.degree=degree
        self.include_bias=include_bias

    def fit_transform(self,X):
        if self.include_bias:
            out=np.ones((len(X),self.degree+1))
            for i in range(self.degree+1):
              out[:,i]=X**i

        else:
            out=np.zeros((len(X),self.degree))
            for i in range(self.degree):
              out[:,i]=X**(i+1)

        return out

Now, for example, if you wanted to create a matrix whose first column is `[0,1,2,3]` and second column is those values squared, you would do this:

In [ ]:
quad=PolynomialFeatures(2)
quad.fit_transform(np.array([0,1,2,3]))

A quadratic prediction has the form intercept + coef[0]·z + coef[1]·z², where z is scaled displacement. It is curved as a function of displacement, but still linear in the coefficients. We have changed the **features**, not the least-squares loss or fitting algorithm.

In [ ]:
quad=PolynomialFeatures(2)
quad_train_feats=quad.fit_transform(scaled_Xtrain)
quad_val_feats=quad.fit_transform(scaled_Xval)

quadratic_mod=LinearRegression()
quadratic_mod.fit(quad_train_feats,ytrain)
quad_train_predictions=quadratic_mod.predict(quad_train_feats)
quad_val_predictions=quadratic_mod.predict(quad_val_feats)

quad_train_feats.shape, quadratic_mod.coef, quadratic_mod.intercept

Inspect one engineered row, then evaluate the fitted quadratic at scaled displacement z = 0.5. This is not raw displacement 0.5.

In [ ]:
quad_train_feats[0], scaled_Xtrain[0]

In [ ]:
quadratic_mod.predict(np.array([[.5,.5**2]]))[0]

## 5. Compare model flexibility visually

Now fit a line and a degree-8 model using exactly the same training observations and the same fitted scaler. Degree 8 produces eight feature columns; LinearRegression adds the ninth, constant column internally.

In [ ]:
line_features=PolynomialFeatures(1)
line_train_feats=line_features.fit_transform(scaled_Xtrain)
line_model=LinearRegression()
line_model.fit(line_train_feats,ytrain)
line_train_predictions=line_model.predict(line_train_feats)

deg8=PolynomialFeatures(8)
deg8_train_feats=deg8.fit_transform(scaled_Xtrain)
deg8_val_feats=deg8.fit_transform(scaled_Xval)
deg8_model=LinearRegression()
deg8_model.fit(deg8_train_feats,ytrain)
deg8_train_predictions=deg8_model.predict(deg8_train_feats)
deg8_val_predictions=deg8_model.predict(deg8_val_feats)

The curves below connect predictions at the training input values, forming polygonal approximations to the fitted polynomials. Sort the input/prediction pairs together **only for plotting**, so the line travels from left to right. Do not sort targets or predictions independently.

Compare where each model misses the overall pattern and where extra flexibility appears to follow individual observations. Curve shape alone does not establish overfitting; next we will check errors on held-out observations.

In [ ]:
order = np.argsort(Xtrain)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
for ax, degree, predictions in zip(
    axes, [1, 2, 8],
    [line_train_predictions, quad_train_predictions, deg8_train_predictions]
):
    ax.scatter(Xtrain, ytrain, label='Training', s=24)
    ax.scatter(Xval, yval, label='Validation', marker='x', s=35)
    ax.plot(Xtrain[order], predictions[order], '-r', label='Model')
    ax.set_title('Degree ' + str(degree))
    ax.set_xlabel('Displacement')
axes[0].set_ylabel('MPG')
axes[0].legend()
plt.tight_layout()
plt.show()

## 6. Measure generalization

RSS adds squared residuals; MSE divides RSS by the number of observations. Training and validation sets have different sizes, so compare their **MSE**, not their raw RSS.

In metrics, row d stores the results for degree d+1. Column 0 is training MSE and column 1 is validation MSE. Each model below is fitted on training data only; validation targets are used only to evaluate predictions.

In [ ]:
metrics=np.zeros((8,2))
for d in range(8):
    poly=PolynomialFeatures(d+1)
    train_feats=poly.fit_transform(scaled_Xtrain)
    val_feats=poly.fit_transform(scaled_Xval)
    model=LinearRegression()
    model.fit(train_feats,ytrain)
    train_predictions=model.predict(train_feats)
    val_predictions=model.predict(val_feats)
    MSEtrain=((train_predictions-ytrain)**2).mean()
    MSEval=((val_predictions-yval)**2).mean()
    metrics[d,:]=[MSEtrain,MSEval]

pd.DataFrame(metrics, index=np.arange(1,9),
             columns=['Training MSE', 'Validation MSE'])

In [ ]:
plt.figure(figsize=(7, 4))
degrees=np.arange(1,9)
plt.plot(degrees,metrics[:,0],'-o',label='Training MSE')
plt.plot(degrees,metrics[:,1],'-s',label='Validation MSE')
plt.xticks(degrees)
plt.xlabel('Polynomial degree')
plt.ylabel('MSE (MPG squared)')
plt.legend()
plt.show()

As degree increases, the set of possible fitted functions grows: a higher-degree model can reproduce a lower-degree one by setting its additional coefficients to zero. With exact least-squares solutions, training MSE therefore cannot increase (small numerical differences are possible).

Validation MSE need not decrease. A model can fit training-specific noise and do worse on unseen observations: **overfitting**. High degree alone is not proof of overfitting, and validation error need not form a perfectly smooth U-shaped curve.

Before running the next cell, find the degree with the lowest validation MSE in the table. Why is one added to argmin?

In [ ]:
best_degree=metrics[:,1].argmin()+1
best_degree

## Check your understanding

- What changes when degree increases: the loss, the solver, or the feature matrix?
- Why do we reuse the training scaler for validation observations?
- Which comparison in the MSE table is evidence that added flexibility hurt generalization?
- Why is the smallest training MSE not a sufficient reason to choose a degree?
- Why would reporting the best validation MSE as a final test result be optimistic?

Complete **5. Polynomial Regression and Overfitting** in Machine Learning Lab. Its notebook labs use small deterministic arrays, so their answers do not depend on the downloaded car dataset. The next assignment, Homework 6, introduces gradient descent; later regularization will offer another way to control flexibility.